# IT4060 - HPC Failure Prediction

## Notebook: Data Cleaning

This notebook loads the interim exports from `01_data_exploration.ipynb`, performs additional cleaning, validates the cleaned tables, and writes cleaned datasets back to `data/interim/`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

In [2]:
PROJECT_NAME = 'IT4060-ML-Assignment-HPC-Failure-Prediction'

def find_project_root():
    cwd = Path.cwd().resolve()
    home = Path.home().resolve()
    desktop = home / 'Desktop'
    candidate_roots = [cwd, *cwd.parents, home, desktop, desktop / 'Manilka' / 'ML_Assignment']
    seen = set()

    for base in candidate_roots:
        for candidate in (base, base / PROJECT_NAME):
            if candidate in seen or not candidate.exists():
                continue
            seen.add(candidate)
            if (candidate / 'data' / 'interim').exists():
                return candidate

    raise FileNotFoundError('Could not locate the project root with data/interim.')

project_root = find_project_root()
interim_dir = project_root / 'data' / 'interim'

failure_interim_path = interim_dir / 'failure_system20_nodes_0_255.csv'
usage_nodes_interim_path = interim_dir / 'usage_node_events.csv.gz'

failure_clean_path = interim_dir / 'failure_system20_clean.csv'
usage_nodes_clean_path = interim_dir / 'usage_node_events_clean.csv.gz'

required_files = [failure_interim_path, usage_nodes_interim_path]
missing_files = [path.name for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Missing interim files: ' + ', '.join(missing_files) + '. Run 01_data_exploration.ipynb first.'
    )

print(f'Working directory: {Path.cwd()}')
print(f'Project root: {project_root}')
print(f'Interim directory: {interim_dir}')

Working directory: D:\Projects\IT4060-ML-Assignment-HPC-Failure-Prediction\notebooks
Project root: D:\Projects\IT4060-ML-Assignment-HPC-Failure-Prediction
Interim directory: D:\Projects\IT4060-ML-Assignment-HPC-Failure-Prediction\data\interim


In [3]:
failure_df = pd.read_csv(
    failure_interim_path,
    parse_dates=['failure_start', 'failure_end'],
    low_memory=False,
)

usage_nodes_df = pd.read_csv(
    usage_nodes_interim_path,
    parse_dates=['usage_time', 'submit_time', 'dispatch_time'],
    low_memory=False,
)

overview = pd.DataFrame([
    {'dataset': 'failure_interim', 'rows': len(failure_df), 'columns': failure_df.shape[1]},
    {'dataset': 'usage_node_events_interim', 'rows': len(usage_nodes_df), 'columns': usage_nodes_df.shape[1]},
])

display(overview)
display(failure_df.head())
display(usage_nodes_df[['usage_time', 'node', 'user_id', 'num_procs', 'total_time', 'status']].head())

,dataset,rows,columns
0,failure_interim,2124,27
1,usage_node_events_interim,1281928,18


,System,machine type,nodes,procstot,procsinnode,nodenum,nodenumz,node install,node prod,node decom,fru type,mem,cputype,memtype,num intercon,purpose,failure_start,failure_end,Down Time,Facilities,Hardware,Human Error,Network,Undetermined,Software,Same Event,failure_category
0,20,cluster,512.0,2048.0,4.0,194.0,194,1-Oct,1-Dec,current,part,16.0,2.0,2.0,2.0,compute,2001-12-20 08:00:00,2001-12-26 13:15:00,8955,NaN,CPU,NaN,NaN,NaN,NaN,No,Hardware
1,20,cluster,512.0,2048.0,4.0,251.0,251,1-Oct,1-Dec,current,part,16.0,2.0,2.0,2.0,compute,2001-12-22 23:35:00,2001-12-23 02:28:00,173,NaN,CPU,NaN,NaN,NaN,NaN,No,Hardware
2,20,cluster,512.0,2048.0,4.0,197.0,197,1-Oct,1-Dec,current,part,16.0,2.0,2.0,2.0,compute,2001-12-23 00:52:00,2001-12-23 02:28:00,96,NaN,CPU,NaN,NaN,NaN,NaN,No,Hardware
3,20,cluster,512.0,2048.0,4.0,234.0,234,1-Oct,1-Dec,current,part,16.0,2.0,2.0,2.0,compute,2001-12-23 06:27:00,2001-12-23 08:30:00,123,NaN,CPU,NaN,NaN,NaN,NaN,No,Hardware
4,20,cluster,512.0,2048.0,4.0,178.0,178,1-Oct,1-Dec,current,part,16.0,2.0,2.0,2.0,compute,2001-12-24 06:00:00,2001-12-26 13:15:00,3315,NaN,CPU,NaN,NaN,NaN,NaN,No,Hardware


,usage_time,node,user_id,num_procs,total_time,status
0,2003-12-26 12:32:30,222,3405,4,30338.000000,aborted
1,2003-12-26 12:37:22,249,3405,4,817868.000000,aborted
2,2003-12-26 12:41:37,253,3405,4,826427.000000,aborted
3,2003-12-26 12:42:40,146,3052,4,606.000000,aborted
4,2003-12-26 12:50:25,3,15564,4,0.650016,finished


In [4]:
usage_nodes_df['status'] = usage_nodes_df['status'].fillna('missing').astype('string').str.strip().str.lower()

numeric_columns = [
    'user_id', 'num_procs', 'cpu_user_time', 'cpu_system_time', 'total_time',
    'batch_id', 'line_number', 'node', 'node_count'
]
for column in numeric_columns:
    if column in usage_nodes_df.columns:
        usage_nodes_df[column] = pd.to_numeric(usage_nodes_df[column], errors='coerce')

usage_nodes_df = usage_nodes_df.loc[
    usage_nodes_df['usage_time'].notna() &
    usage_nodes_df['node'].between(0, 255, inclusive='both') &
    usage_nodes_df['batch_id'].notna()
].copy()

usage_nodes_df['queue_wait_seconds'] = (
    usage_nodes_df['dispatch_time'] - usage_nodes_df['submit_time']
).dt.total_seconds()
usage_nodes_df.loc[usage_nodes_df['queue_wait_seconds'] < 0, 'queue_wait_seconds'] = np.nan

usage_nodes_df['cpu_time_per_proc'] = np.where(
    usage_nodes_df['num_procs'] > 0,
    usage_nodes_df['total_time'] / usage_nodes_df['num_procs'],
    np.nan,
)
usage_nodes_df['requested_procs_per_node'] = np.where(
    usage_nodes_df['node_count'] > 0,
    usage_nodes_df['num_procs'] / usage_nodes_df['node_count'],
    np.nan,
)
usage_nodes_df['cpu_user_ratio'] = np.where(
    usage_nodes_df['total_time'] > 0,
    usage_nodes_df['cpu_user_time'] / usage_nodes_df['total_time'],
    np.nan,
)

usage_nodes_df['hour'] = usage_nodes_df['usage_time'].dt.floor('h')
usage_nodes_df['hour_of_day'] = usage_nodes_df['usage_time'].dt.hour
usage_nodes_df['day_of_week'] = usage_nodes_df['usage_time'].dt.dayofweek
usage_nodes_df['is_weekend'] = usage_nodes_df['day_of_week'].isin([5, 6]).astype(int)

status_buckets = ['finished', 'aborted', 'failed', 'killed', 'syskill', 'missing']
for status in status_buckets:
    usage_nodes_df[f'status_{status}'] = (usage_nodes_df['status'] == status).astype(int)
usage_nodes_df['status_other'] = (~usage_nodes_df['status'].isin(status_buckets)).astype(int)

duplicate_usage_rows = usage_nodes_df.duplicated(subset=['batch_id', 'node', 'usage_time', 'line_number']).sum()
usage_nodes_df = usage_nodes_df.drop_duplicates(subset=['batch_id', 'node', 'usage_time', 'line_number']).copy()

duplicate_failure_rows = failure_df.duplicated(subset=['nodenumz', 'failure_start', 'failure_end', 'Hardware', 'Software', 'Undetermined']).sum()
failure_df = failure_df.drop_duplicates(subset=['nodenumz', 'failure_start', 'failure_end', 'Hardware', 'Software', 'Undetermined']).copy()

failure_df['failure_duration_minutes'] = (
    failure_df['failure_end'] - failure_df['failure_start']
).dt.total_seconds() / 60
failure_df['failure_duration_minutes'] = failure_df['failure_duration_minutes'].clip(lower=0)

cleaning_summary = pd.DataFrame([
    {'metric': 'Duplicate usage-node rows removed', 'value': int(duplicate_usage_rows)},
    {'metric': 'Duplicate failure rows removed', 'value': int(duplicate_failure_rows)},
    {'metric': 'Clean usage-node rows', 'value': len(usage_nodes_df)},
    {'metric': 'Clean failure rows', 'value': len(failure_df)},
    {'metric': 'Usage rows with missing queue wait', 'value': int(usage_nodes_df['queue_wait_seconds'].isna().sum())},
])

missing_summary = usage_nodes_df[
    ['queue_wait_seconds', 'cpu_time_per_proc', 'requested_procs_per_node', 'cpu_user_ratio']
].isna().mean().rename('missing_ratio').to_frame()

display(cleaning_summary)
display(missing_summary)
display(usage_nodes_df[['usage_time', 'node', 'queue_wait_seconds', 'cpu_time_per_proc', 'requested_procs_per_node', 'cpu_user_ratio']].head())

,metric,value
0,Duplicate usage-node rows removed,0
1,Duplicate failure rows removed,2
2,Clean usage-node rows,1281928
3,Clean failure rows,2122
4,Usage rows with missing queue wait,12


,missing_ratio
queue_wait_seconds,0.000009
cpu_time_per_proc,0.000000
requested_procs_per_node,0.000000
cpu_user_ratio,0.001744


,usage_time,node,queue_wait_seconds,cpu_time_per_proc,requested_procs_per_node,cpu_user_ratio
0,2003-12-26 12:32:30,222,66.0,7584.500000,4.0,0.997528
1,2003-12-26 12:37:22,249,577.0,204467.000000,4.0,0.997789
2,2003-12-26 12:41:37,253,836.0,206606.750000,4.0,0.997790
3,2003-12-26 12:42:40,146,9.0,151.500000,4.0,0.519802
4,2003-12-26 12:50:25,3,0.0,0.162504,4.0,0.241742


In [5]:
failure_df = failure_df.sort_values(['failure_start', 'nodenumz']).copy()
usage_nodes_df = usage_nodes_df.sort_values(['usage_time', 'node', 'batch_id']).copy()

failure_df.to_csv(failure_clean_path, index=False)
usage_nodes_df.to_csv(usage_nodes_clean_path, index=False, compression='gzip')

export_summary = pd.DataFrame([
    {'file_name': failure_clean_path.name, 'rows': len(failure_df), 'description': 'Cleaned System 20 failure records'},
    {'file_name': usage_nodes_clean_path.name, 'rows': len(usage_nodes_df), 'description': 'Cleaned usage node-event rows with derived cleaning fields'},
])

display(export_summary)
print('Cleaned interim files written successfully.')

,file_name,rows,description
0,failure_system20_clean.csv,2122,Cleaned System 20 failure records
1,usage_node_events_clean.csv.gz,1281928,Cleaned usage node-event rows with derived cle...


Cleaned interim files written successfully.


## Next Step

Use [03_feature_engineering.ipynb](./03_feature_engineering.ipynb) to load these cleaned interim tables and build the final model-ready feature dataset.